In [2]:
import numpy as np
from scipy import stats
from math import ceil
from pathlib import Path
import matplotlib.pyplot as plt
from contact_plots import get_data_from_files, get_combined_data_files_from_dir, do_chi_square_test
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder)

In [ ]:
infra_name = 'base_case'
base = 'nn_nts_bvd'
other_case = 'n_nts_bvd'

data_name = '1.5_customer_staff_contacts_over_agents'

base_array = infra_and_measure_name_2_data[infra_name][base][data_name]
compare_array = infra_and_measure_name_2_data[infra_name][other_case][data_name]


In [ ]:
base_values, base_counts = np.unique(base_array, return_counts=True)
base_counts = base_counts/sum(base_counts)
base_counts_dict = dict(zip(base_values, base_counts))
compare_values, compare_counts = np.unique(compare_array, return_counts=True)
compare_counts = compare_counts/sum(compare_counts)
compare_counts_dict = dict(zip(compare_values, compare_counts))
values = np.unique(np.concatenate((base_values, compare_values)))

print(base_values)
print(compare_values)
print(values)

base_counts = [base_counts_dict[value] if value in base_counts_dict else 0 for value in values]
compare_counts = [compare_counts_dict[value] if value in compare_counts_dict else 0 for value in values]


print(base_counts)
print(compare_counts)

if len(values) == 1:
    print(1)

res = stats.chisquare(compare_counts, base_counts)

print(res)

In [ ]:
base_values, base_counts = np.unique(base_array, return_counts=True)
base_counts = base_counts
base_counts_dict = dict(zip(base_values, base_counts))
compare_values, compare_counts = np.unique(compare_array, return_counts=True)
compare_counts = compare_counts/sum(compare_counts)*sum(base_counts)
compare_counts_dict = dict(zip(compare_values, compare_counts))
values = np.unique(np.concatenate((base_values, compare_values)))

print(base_values)
print(compare_values)
print(values)

base_counts = [base_counts_dict[value] if value in base_counts_dict else 0 for value in values]
compare_counts = [compare_counts_dict[value] if value in compare_counts_dict else 0 for value in values]


print(base_counts)
print(compare_counts)

if len(values) == 1:
    print(1)

res = stats.chisquare(compare_counts, base_counts)

print(res)

In [ ]:
combined_data_files = get_combined_data_files_from_dir(data_folder)
for combined_data_file in combined_data_files:
    print(f'{combined_data_file.stem}\n')
    num_data = np.load(combined_data_file, allow_pickle=True)
    arrays_dict = num_data['arrays'].item()
    indices_dict = num_data['indices'].item()
    for data_name, indices in indices_dict.items():
        if 'contacts_over_agents' not in data_name:
            continue
        p_values = []
        array = arrays_dict[data_name]
        base_array = array[:indices[-1]]
        for ii in range(2, len(indices)+1):
            compare_array = array[:indices[-ii]]
            base_values, base_counts_org = np.unique(base_array, return_counts=True)
            base_counts = base_counts_org
            base_counts_dict = dict(zip(base_values, base_counts))
            compare_values, compare_counts_org = np.unique(compare_array, return_counts=True)
            compare_counts = compare_counts_org/sum(compare_counts_org)*sum(base_counts)
            compare_counts_dict = dict(zip(compare_values, compare_counts))
            values = np.unique(np.concatenate((base_values, compare_values))) 
            base_counts = [base_counts_dict[value] if value in base_counts_dict else 0 for value in values]
            compare_counts = [compare_counts_dict[value] if value in compare_counts_dict else 0 for value in values]
            if len(values) == 1:
                p_values.append(1.0)
                continue

            res = stats.power_divergence(compare_counts, base_counts)
            if res.pvalue < 0.99:
                base_norm = base_counts_org/sum(base_counts_org)
                compare_norm = compare_counts_org/sum(compare_counts_org)
                rel_diff = (compare_norm - base_norm)/base_norm*100
                #print(f'{base_counts} - {compare_counts_org}    ---     {base_norm} - {compare_norm}     ---    {rel_diff}')
            p_values.append(res.pvalue)
        
        print(f'{data_name}: ', end='')
        print_warn = False
        for p_value in p_values:
            print(f'{p_value:.3f}, ', end='')
            if p_value < 0.99:
                print_warn = True
        if print_warn:
            print('\n -------- Not converged!')
        else:
            print('')
        
    print(f'{"="*40}\n')
            

In [ ]:
combined_data_files = get_combined_data_files_from_dir(data_folder)
combs_processed = []
for combined_data_file in combined_data_files:
    for other_combined_data_file in combined_data_files:
        if combined_data_file == other_combined_data_file or (combined_data_file, other_combined_data_file) in combs_processed or combined_data_file.stem[:6] != other_combined_data_file.stem[:6]:
            continue
        combs_processed.append((combined_data_file, other_combined_data_file))
            
        print(f'{combined_data_file.stem} - {other_combined_data_file.stem}\n')
        num_data = np.load(combined_data_file, allow_pickle=True)
        arrays_dict = num_data['arrays'].item()

        other_num_data = np.load(other_combined_data_file, allow_pickle=True)
        other_arrays_dict = other_num_data['arrays'].item()
        
        for data_name, base_array in arrays_dict.items():
            if 'customer_staff_contacts_over_agents' not in data_name:
                continue
            compare_array = other_arrays_dict[data_name]
            base_values, base_counts_org = np.unique(base_array, return_counts=True)
            base_counts = base_counts_org
            base_counts_dict = dict(zip(base_values, base_counts))
            compare_values, compare_counts_org = np.unique(compare_array, return_counts=True)
            compare_counts = compare_counts_org/sum(compare_counts_org)*sum(base_counts_org)
            compare_counts_dict = dict(zip(compare_values, compare_counts))
            values = np.unique(np.concatenate((base_values, compare_values))) 
            base_counts = [base_counts_dict[value] if value in base_counts_dict else 0 for value in values]
            compare_counts = [compare_counts_dict[value] if value in compare_counts_dict else 0 for value in values]
            print(f'{base_counts} - {compare_counts}')
            
            if len(values) == 1:
                p_value = 1.0
            else:
                res = stats.power_divergence(compare_counts, base_counts)
                p_value = res.pvalue
                
            print(f'{data_name}: {p_value:.4f}')
        
           
        print(f'{"="*40}\n')

In [5]:
a = [np.array([1,2,3]), np.array([2,3]), np.array([1,3])]

np.unique(np.concatenate(a)).tolist()

[1, 2, 3]